# IOAI — 2026 Contest Molecular Energy Dipole (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-molecular-energy-dipole/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Molecular Energy & Dipole 예측 (APOAI 2026)

분자의 **원자 좌표(xyz)** 만으로 **에너지**와 **쌍극자모멘트 크기**를 예측한다. 학습 분자는 단 **200개**
(QM9 부분). 원자는 H/C/N/O/F, 최대 50원자, 중성·바닥상태. 에너지와 쌍극자 크기는 **평행이동·회전에 불변**.

`data/train/molecule_N/coordinate.xyz` + `data/train/training_ref.csv`(energy, dipole)로 학습하고
`data/test/molecule_N/coordinate.xyz` 를 예측해 `submission.csv`(`molecule #, energy, dipole moment`)를 만든다.
점수 = 에너지 MRE 점수 + 쌍극자 MAE 점수(각 최대 0.5).

In [ ]:
import pandas as pd, numpy as np, glob, os
ref = pd.read_csv("data/train/training_ref.csv")           # molecule #, energy, dipole moment
ref.columns = ["mol", "energy", "dipole"]
test_ids = pd.read_csv("data/test/test_list.csv")["molecule #"].tolist()
PROP = {"H":(1,2.20),"C":(6,2.55),"N":(7,3.04),"O":(8,3.44),"F":(9,3.98)}   # (nuclear charge, electronegativity)
ATOMS = "HCNOF"

def parse_xyz(path):
    ls = open(path).read().splitlines(); n = int(ls[0])
    syms, xyz = [], []
    for l in ls[2:2+n]:
        p = l.split(); syms.append(p[0]); xyz.append([float(v) for v in p[1:4]])
    return syms, np.array(xyz)
print("train molecules", len(ref), "| test molecules", len(test_ids))

In [ ]:
# 베이스라인: 학습셋 평균 에너지·쌍극자를 모두에 예측 (점수 낮음)
me, md_ = ref.energy.mean(), ref.dipole.mean()
import csv
rows = [{"molecule #": m, "energy": me, "dipole moment": md_} for m in test_ids]
pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)